# Module 1: The Inference Stack and Runtime Landscape

You are here to own the layer your agents run on: a model you serve yourself with [vLLM](https://docs.vllm.ai) on a dedicated [Akamai Cloud GPU](https://www.linode.com/products/gpu/), instead of renting tokens from a hosted API. This module is the on-ramp. You connect to the GPU you own and send a first request. You point a small agent at it, the [Akamai Cloud Solutions Architect agent](https://github.com/akamai-developers/akamai-workshop-solution-architect-agent/tree/main/agent) which you will deploy in Module 10. You compare the cost of renting against owning with your own numbers. Then you place vLLM among the other runtimes and trace one request end to end, so you can name where its time goes before you start measuring it.

## Learning objectives
- Resolve your connection settings from the environment and reach your own vLLM endpoint
- Point a small agent at your endpoint and confirm its model is the GPU you own
- Compare a hosted API call against your self-hosted server on cost, data path, and control
- Review vLLM among other runtimes (SGLang, TensorRT-LLM, llama.cpp) and understand why this workshop runs it
- Trace one request from client to streamed tokens to see time to first token and time per output token
- Read the vLLM metrics endpoint, the instrument you use in every later module
- Run a few agent steps and see that an agent is many requests, where the slow step sets the pace

## Prerequisites
- Path A (hosted workshop): your environment is already running, signed in with your access card
- Path B (bring your own): infrastructure stood up with the `akamai-workshop-platform` repo, variables exported
- This is the first module, so start here
- About 12 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [SGLang](https://github.com/sgl-project/sglang) &middot; [TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/overview.html) &middot; [llama.cpp](https://github.com/ggml-org/llama.cpp) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat)

## Inference stack design basics

A request to a self-hosted model passes through a few layers. Knowing them tells you where to look when something is slow or expensive.

- The runtime is the server that loads the model and answers requests. This workshop runs vLLM. Others are SGLang, TensorRT-LLM, and llama.cpp.
- The API surface is how you talk to it. vLLM speaks the OpenAI API, so your client code points at it with one change, the base URL.
- The scheduler decides which requests run now and which wait, and it batches requests together to use the GPU well.
- The KV cache holds the running attention state for every in-flight request. It lives in the same GPU memory as the model weights.
- The metrics endpoint reports what the server is doing right now: queue depth, cache use, and latency. You read it all day.

![Inside the Akamai LKE cluster: your JupyterLab notebooks call a vLLM Service on a dedicated GPU node in your namespace, and read its metrics endpoint](images/01_inference_stack_architecture.png)

## The thread: an agent is a loop

A chatbot is one request: one prefill, one decode. An agent is a loop. Every step it re-sends a growing prompt (the system prompt, the tool schemas, and every Thought, Action, and Observation so far), decodes a reasoning trace, then calls a tool and goes again. So an agent is many requests on the server you own, and the inference layer you build in these modules is the agent's latency and cost budget.

Hold five consequences in mind. Each one lands on a later module:

- Context grows every turn and never shrinks, so the 2048 cap is a task-length ceiling (Module 2).
- The system prompt and tool schemas get re-prefilled every step, unless you cache them (Module 3).
- An orchestrator fanning out to sub-agents spikes concurrent load and can preempt (Module 3).
- The agent's wall clock is the sum of its steps, so one slow step stalls the chain (this module).
- The model thinks before it answers, and thinking tokens cost decode time (Module 4).

You build a small version of this loop in `common/agent_loop.py` and point it at your own vLLM. Every module runs it against one of those five.

![A chatbot is one request; an agent is a loop that re-prefills a growing prompt every step](images/01_agent_loop.png)


## 1. Setup

Install the packages this module needs. We reinstall here so this notebook stands on its own. `kubectl` is already on your PATH in the workshop environment.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

## 2. Reach the server you own

`common/config.py` reads your connection details from the environment and fills in defaults. `print_settings()` shows the resolved values, and never prints the key value, only whether one is set. Then list your pods to see your vLLM running, confirm it holds a GPU, and send one chat completion through it.

In [ ]:
# Make the repo's common/ package importable, then print the settings from your environment.
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

from common.config import print_settings, build_client

settings = print_settings()

**What you should see:** your `VLLM_HOST`, the derived metrics URL, `MODEL_NAME`, and `NAMESPACE`. On Path A these are filled in for you. If `VLLM_HOST` still shows the default and you are off-cluster on Path B, set the variables and restart the kernel.

In [ ]:
# Requires a live cluster. List the pods in your namespace; expect your vLLM pod Running.
ns = settings.namespace
!kubectl get pods -n {ns}

**What you should see:** your vLLM pod with `STATUS` `Running` and `READY` `1/1`. If it is `Pending`, give it a minute and re-run. A connection or permission error means `KUBECONFIG` is not set right.

In [ ]:
# Your kubeconfig is namespace-scoped, so `kubectl get nodes` is Forbidden by design.
# Confirm YOUR vLLM pod requested a GPU, and see the node it landed on.
ns = settings.namespace
!kubectl get pods -n {ns} -l app=vllm -o custom-columns=POD:.metadata.name,NODE:.spec.nodeName,GPU:.spec.containers[0].resources.limits.'nvidia\.com/gpu'

**What you should see:** one row for your vLLM pod with `GPU` `1` and the node it runs on. A GPU of `<none>` means the pod did not request one. An empty list means the pod is not up yet, so check the previous cell.

In [ ]:
# Build an OpenAI client pointed at your endpoint and send one chat completion.
client = build_client(settings)
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "In one sentence, what is an inference server?"}],
    max_tokens=64,
    temperature=0.0,
)
print(resp.choices[0].message.content)

**What you should see:** one sentence of generated text from a server you control. A connection error means `VLLM_HOST` is wrong or the pod is not ready. A 404 on the model means `MODEL_NAME` does not match what the server loaded.

## 3. The agent on your endpoint

The workshop is named for the agent, so meet it now. The agent is the `openai` client wrapped with a system prompt that gives it a job. This is the Akamai Cloud Solutions Architect agent you deploy as a real service in Module 10. Here it is one function. Point it at your vLLM, ask it one question, and ask it what powers it.

In [ ]:
# A minimal agent: a system prompt plus a call to the model you own. No framework.
SYSTEM_PROMPT = (
    "You are the Akamai Cloud Solutions Architect agent. You help developers with "
    "Akamai Cloud and the Kubernetes cluster you run in. You are tactical and concise, "
    "developer to developer. In scope: Compute, LKE, Object Storage, Cloud networking, "
    "GPUs, and AI inference. You run on self-hosted inference served by vLLM. If asked "
    "what powers you, say so."
)

def agent(message):
    resp = client.chat.completions.create(
        model=settings.model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        max_tokens=200,
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

print(agent("What model and inference server are you running on?"))

**What you should see:** an answer that names self-hosted vLLM. The agent's brain is the GPU in your namespace, not a rented API. You spend the rest of the workshop making that brain faster and cheaper, then you deploy this agent on top of it in Module 10.

## 4. An agent is many requests on your endpoint

The agent above answered in one shot. A real agent loops: it calls the model, reads a tool result, and calls again. Run a few steps from `common/agent_loop.py` and watch each step arrive as its own request with its own time to first token. The agent's wall clock is the sum of those steps, so the slowest step sets the pace.

In [ ]:
# Requires a live vLLM endpoint. Run 4 agent steps and time each as its own request.
from common import agent_loop
records = agent_loop.run(steps=4, obs_tokens=200, max_tokens=120)

print(f"{'step':>4} {'ttft_ms':>8} {'prompt_tok':>11} {'out_tok':>8}")
for r in records:
    print(f"{r['step']:>4} {str(r.get('ttft_ms')):>8} {str(r.get('prompt_tokens')):>11} {str(r.get('completion_tokens')):>8}")
print(f"\nthe agent made {len(records)} model calls; its wall clock is the sum of their steps")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
ok = [r for r in records if r["ok"] and r["ttft_ms"]]
plt.figure(figsize=(7, 3))
plt.bar([r["step"] for r in ok], [r["ttft_ms"] for r in ok], color="#6c5ce7")
plt.xlabel("agent step"); plt.ylabel("TTFT (ms)")
plt.title("Each agent step is its own request"); plt.tight_layout()

**What you should see:** four steps, each its own request with its own TTFT, and `prompt_tokens` climbing each step (that is the context tax you measure in Module 2). The agent's total time is the sum of the steps, so a single slow step, one that queues behind a big prefill or gets preempted under load, stalls the whole chain. For agents you optimize the slow tail (p99), not the average. The two biggest levers come next: prefix caching to cut the per-step prefill (Module 3), and the KV budget to avoid preemption (Modules 2 and 9).

## 5. Renting versus owning, with your own numbers

Inference is the same operation either way: a prompt goes in, tokens come out. What differs is the path and the price. Send a prompt to your server and read the `usage` object, the token counts a provider would bill. Then put your real volume into the cost cell and see where owning passes renting.

In [ ]:
# Send one prompt to the server you control and read the token usage a provider would bill.
prompt = "Explain what an LLM inference server does, in two sentences."
start = time.time()
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0.0,
)
print(resp.choices[0].message.content)
print(f"\nself-hosted latency: {time.time() - start:.2f}s")
print("usage:", resp.usage)

**What you should see:** a two-sentence answer, a latency in the low seconds, and a `usage` object. Those token counts are the unit of a hosted bill. On your own server they cost nothing per request.

In [ ]:
# Estimate the monthly bill both ways at YOUR volume. Edit these to match your workload.
requests_per_day = 200_000
input_tokens_per_request = 500
output_tokens_per_request = 300

# Hosted prices in dollars per million tokens. Replace with your provider's.
hosted_input_price = 0.15
hosted_output_price = 0.60

# Owned: one dedicated GPU instance, billed by the hour no matter the token count.
gpu_hourly = 1.50   # your Akamai Cloud GPU plan price per hour

in_tokens = requests_per_day * input_tokens_per_request * 30
out_tokens = requests_per_day * output_tokens_per_request * 30
hosted_monthly = (in_tokens / 1e6) * hosted_input_price + (out_tokens / 1e6) * hosted_output_price
owned_monthly = gpu_hourly * 24 * 30

print(f"hosted bill (rented): ${hosted_monthly:,.0f} / month, and it grows with every token")
print(f"owned GPU (fixed)   : ${owned_monthly:,.0f} / month, flat no matter the volume")
verdict = "owning is cheaper at this volume" if owned_monthly < hosted_monthly else "renting is cheaper at this volume"
print(f"=> {verdict}")

**What you should see:** a hosted bill that scales with every token, and a flat owned cost. There is a crossover, and you just found it for your volume. Module 9 turns measured throughput into a cost per million tokens at a latency target.

## 6. What a price tag hides

Cost is the obvious axis. Three others matter as much in production.

- **Data residency.** Your self-hosted request never left the cluster. For regulated data or anything under a contractual boundary, that is the line between compliant and not.
- **Rate limits.** A hosted provider sets your throughput and can throttle you during a launch. On your own server the only ceiling is the card you provisioned, and you watch it coming in the metrics.
- **Control.** You pick the model, the precision, the context length, and the batching policy. You tune for your traffic instead of accepting defaults. That is the rest of this workshop.

## 7. The runtime landscape

vLLM is one of several runtimes. Knowing where it sits tells you what you are choosing and what you could switch to.

- **vLLM**: the broad default for GPU serving. PagedAttention KV cache, continuous batching, wide model and quantization support, OpenAI-compatible server. Best when you want to run many models fast with no per-model compile step.
- **SGLang**: best when requests share a long prefix, like multi-turn chat, agents, and RAG. Its prefix cache reuses the KV cache across requests, and it is fast at structured output.
- **TensorRT-LLM**: best at peak throughput and latency on NVIDIA hardware. NVIDIA only, and you pay an upfront tuning cost, so it fits stable models in long-running production.
- **llama.cpp**: best for local, edge, and CPU-first inference. Runs heavily quantized models from a single file on a laptop or an Arm server.

All four expose an OpenAI-compatible server, so your client code moves between them. This workshop runs vLLM because it is the common denominator, and it carries the batching and KV cache ideas you measure all day. Ask your own server which models it serves.

In [ ]:
# Ask the server which models it serves. The OpenAI-compatible surface is why your code does not change.
import requests
root = settings.vllm_host.rstrip("/").removesuffix("/v1")
data = requests.get(
    f"{root}/v1/models",
    headers={"Authorization": f"Bearer {settings.api_key}"},
    timeout=10,
).json()
ids = [m["id"] for m in data.get("data", [])]
print("served models      :", ids)
print("matches MODEL_NAME :", settings.model_name in ids)

**What you should see:** your model id, and a match against `MODEL_NAME`. This is the same `/v1` surface SGLang, TensorRT-LLM, and llama.cpp expose, which is why switching runtimes does not rewrite your client.

A few terms to carry forward, each proven later:
- Continuous batching: you watch it form in Module 3, and Omer drives it to saturation in Module 8.
- KV cache and PagedAttention: you build the KV cache by hand in Module 3.
- Attention kernels and FlashAttention: Omer covers them in Module 6.
- Quantization: Omer covers it in Module 5.

## 8. Trace one request, end to end

Follow a single request through the server. The client sends a prompt. The scheduler admits it. Prefill reads the whole prompt in one pass and fills the KV cache. Decode then generates one token at a time and streams each back. Two numbers name the two phases: time to first token is prefill, time per output token is decode. Stream one request and measure both.

In [ ]:
# Stream one request, time the first token (prefill) and the gaps between tokens (decode).
start = time.time()
first = None
stamps = []
stream = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "List the numbers 1 through 50, one per line."}],
    max_tokens=120,
    temperature=0.0,
    stream=True,
)
for chunk in stream:
    if not chunk.choices or not chunk.choices[0].delta.content:
        continue
    now = time.time()
    if first is None:
        first = now
    stamps.append(now)

ttft = (first - start) if first else 0.0
gaps = [b - a for a, b in zip(stamps, stamps[1:])]
tpot = (sum(gaps) / len(gaps)) if gaps else 0.0
print(f"time to first token (prefill) : {ttft * 1000:.0f} ms")
if tpot:
    print(f"time per output token (decode): {tpot * 1000:.1f} ms  ({1 / tpot:.0f} tokens/s)")

**What you should see:** a TTFT in the low hundreds of milliseconds and a TPOT of a few to tens of milliseconds. You measured the request lifecycle on your own server. Module 3 explains, from first principles, why prefill is fast per token and decode is the slow part.

## Things to know

- **The OpenAI-compatible surface is why this works.** Your agent code does not change when the server becomes yours. Only the base URL does.
- **The notebook pod has no GPU.** Inference runs in the vLLM pod on a GPU node. You drive and measure it over the network, which is how you operate it in production.
- **Your kubeconfig is scoped.** `kubectl` sees only your namespace. You cannot touch a neighbor's vLLM, and they cannot touch yours.
- **The metrics URL is derived, not separate.** It is `VLLM_HOST` with `/metrics` in place of `/v1`. Module 3 starts reading it for real.

> NOTE: To run the hosted comparison live, set `HOSTED_API_KEY` and a base URL, then call a hosted model in section 4. The lesson is who owns the server, not which model is smarter.

## Try it yourself

**Find your crossover.** Edit `requests_per_day` and the prices in section 5 until the hosted bill passes the cost of a dedicated GPU. That volume is where owning starts to pay. **Stretch:** add a second, larger model's prices and see how much sooner the line crosses.

**Give the agent a real question.** Ask the section 3 agent something in scope (an LKE or GPU question) and something out of scope, and see how the system prompt holds the line.

**Watch the prompt change TTFT.** Send a very short prompt and a very long one in section 8 and compare the time to first token. Prefill grows with prompt length.

In [ ]:
# Change these two lines, then run the cell.
your_volume = 200_000     # your real requests per day
your_gpu_hourly = 1.50    # your Akamai Cloud GPU plan price per hour

inp = your_volume * input_tokens_per_request * 30
out = your_volume * output_tokens_per_request * 30
hosted = (inp / 1e6) * hosted_input_price + (out / 1e6) * hosted_output_price
owned = your_gpu_hourly * 24 * 30
print(f"hosted ${hosted:,.0f}/mo vs owned ${owned:,.0f}/mo")

## Summary

- You resolved your settings from the environment and reached a vLLM endpoint you control.
- You pointed a small agent at it and saw its model is the GPU you own.
- You compared renting against owning on cost, data path, and control, and found the crossover for your volume.
- You placed vLLM among the runtimes and traced one request, naming time to first token (prefill) and time per output token (decode).

## Next

**Module 2: Units and the Memory Budget.** You can reach the server, but you have not yet counted what a model costs in memory. Next you size the weights, the KV cache, and the budget that decides how many users fit on one card, all from arithmetic.